### Importar bibliotecas

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report

import category_encoders as ce

from sklearn.model_selection import train_test_split
import pickle
import os

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dropout, MaxPooling1D, Flatten, Dense
from sklearn.utils.class_weight import compute_class_weight


# verificar placa de video
from tensorflow.python.client import device_lib

# Lista dispositivos detalhados
devices = device_lib.list_local_devices()

for device in devices:
    if device.device_type == 'GPU':
        print(f"Nome detalhado da GPU: {device.physical_device_desc}")


Nome detalhado da GPU: device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


### Leitura da base

#### Definir dados

In [23]:
# definir caminhos
pathDataTrain = "./files/csv/01-12"
pathDataTest = "./files/csv/03-11"
pathIntermediary = "./files/csv/intermediary"
pathPreprocess = "./files/csv/analytic"


# definir bases de dados geral
allDataBases = ["DrDoS_DNS", "DrDoS_LDAP", "DrDoS_MSSQL", "DrDoS_NetBIOS", "DrDoS_NTP", "DrDoS_SNMP", "DrDoS_SSDP", "DrDoS_UDP", "Syn", "TFTP", "UDPLag"]
allDataBasesTest = ["LDAP", "MSSQL", "NetBIOS", "Syn", "UDP", "UDPLag"]

# definir bases de dados de acordo com o artigo
first_part = ["MSSQL", "Portmap", "Syn", "UDP", "UDPLag"]
second_part = ["LDAP", "MSSQL", "NetBios"]

columns_for_remove = ['Unnamed: 0', 'Source IP', 'Destination IP', 'Flow ID', 'Timestamp', 'Source Port', 'Destination Port']

### Pré-Processamento

#### Definir funções

##### Funções de análise e verificação

In [4]:
# verificar a quantidade de valores infinitos em cada coluna numérica do dataframe
def verify_null_values(dataframe):
    null_columns = dataframe.columns[dataframe.isnull().any()]
    null_data = dataframe[null_columns].isnull().sum()
    
    # Adicionar coluna com total de dados (count não nulos) da coluna original
    #total_data = dataframe[null_columns].count()
    total_data = len(dataframe[null_columns].index)
    null_data = null_data.to_frame(name='Qtd null')
    null_data['Qtd Total'] = total_data
    null_data['Percent null (%)'] = (null_data['Qtd null'] / null_data['Qtd Total']) * 100
    
    return null_data

# verificar a quantidade de valores infinitos em cada coluna numérica do dataframe
def verify_infinite_values(dataframe):
    num_tmp_df = dataframe.select_dtypes(include=[np.number])
    infinite_columns = num_tmp_df.columns[np.isinf(num_tmp_df).any()]
    infinite_data = num_tmp_df[infinite_columns].isin([np.inf, -np.inf]).sum()
    
    # Adicionar coluna com total de dados (count não nulos) da coluna original
    total_data = len(num_tmp_df[infinite_columns].index)
    infinite_data = infinite_data.to_frame(name='Qtd Inf')
    infinite_data['Qtd Total'] = total_data
    infinite_data['Percent Inf (%)'] = (infinite_data['Qtd Inf'] / infinite_data['Qtd Total']) * 100
    
    return infinite_data
    
# verificar a quantidade de valores negativos em cada coluna numérica do dataframe
def verify_negative_values(dataframe):    
    num_tmp_df = dataframe.select_dtypes(include=[np.number])
    negative_columns = num_tmp_df.columns[(num_tmp_df < 0).any()]
    negative_data = num_tmp_df[num_tmp_df.loc[:,negative_columns] < 0].count()[negative_columns]
            
    # Adicionar coluna com total de dados (count não nulos) da coluna original
    total_data = len(num_tmp_df[negative_columns].index)
    negative_data = negative_data.to_frame(name='Qtd negative')
    negative_data['Qtd Total'] = total_data
    negative_data['Percent neg. (%)'] = (negative_data['Qtd negative'] / negative_data['Qtd Total']) * 100

    return negative_data

# verificar a quantidade de valores zero em cada coluna numérica do dataframe
def verify_zero_values(dataframe):
    num_tmp_df = dataframe.select_dtypes(include=[np.number])
    zero_columns = num_tmp_df.columns[(num_tmp_df == 0).any()]
    zero_data = num_tmp_df[num_tmp_df.loc[:,zero_columns] == 0].count()[zero_columns]
    
    # Adicionar coluna com total de dados (count não nulos) da coluna original
    total_data = len(num_tmp_df[zero_columns].index)
    zero_data = zero_data.to_frame(name='Qtd zero')
    zero_data['Qtd Total'] = total_data
    zero_data['Percent zero (%)'] = (zero_data['Qtd zero'] / zero_data['Qtd Total']) * 100
    
    # return infinite_data
    return zero_data

# realizar a impressão dos dados analíticos 
def print_analytic(dataframe, message=""):
    
    negative_data = verify_negative_values(dataframe)
    infinity_data = verify_infinite_values(dataframe)
    null_data = verify_null_values(dataframe)
    
    tmp = verify_zero_values(dataframe)
    zero_data = tmp[tmp['Percent zero (%)'] == 100].index

    print(message)
    print("Valores negativos: ", len(negative_data))
    print("Valores infinitos: ", len(infinity_data))
    print("Valores nulos: ", len(null_data))
    print("Colunas com valores zero: ", len(zero_data))
    
    print("--------------------------------------------------") 

# realizar a impressão dos dados analíticos em arquivo
def print_analytic_all_data(dataframe, arq, message=""):
    
    negative_data = verify_negative_values(dataframe)
    infinity_data = verify_infinite_values(dataframe)
    null_data = verify_null_values(dataframe)
    
    tmp = verify_zero_values(dataframe)
    zero_data = tmp[tmp['Percent zero (%)'] == 100]

    
    with open(arq, "a", encoding="utf-8") as f:
        f.writelines("==================================================\n") 
        f.writelines("==================================================\n") 
        f.writelines(message + "\n")
        f.writelines("==================================================\n")         
        f.writelines("==================================================")                 

        
        f.writelines("\n\nValores negativos: \n" + negative_data.to_string())
        f.writelines("\n\nValores infinitos: \n" + infinity_data.to_string())
        f.writelines("\n\nValores nulos: \n" + null_data.to_string())
        f.writelines("\n\nColunas com valores zero: \n" + zero_data.to_string())
        
        f.writelines("--------------------------------------------------\n\n\n\n") 
 

##### Funções auxiliares para tratar a base de dados

In [22]:
def drop_database_null_infinity_values(dataBase):
    # Definir dados
    result  = dataBase
     
    # Remover linhas com valores nulos
    result = result.dropna()

    # Remover linhas com valores infinitos
    result = result[~result.isin([np.inf, -np.inf]).any(axis=1)]
    
    return result   

# Método para tratar os valores infinitos e nulos das colunas passadas do dataframe
def remove_inf_and_null(database, column, analysis_method="auto", important_column=False, drop_column=True):
    
    """ _summary_
    
    Args: 
        database (table): base de dados que se deseja tratar
        column (str): nome da coluna a ser tratada
        analysis_method (str): avg - média, drop - retirar linha, auto - automático, zero - substituir por zero
        important_column (bool): - irá retirar toda a linha se mais de 5% dos dados da coluna for 'inf' ou 'null'
        drop_column (bool): caso a coluna seja maioria 'inf' ou 'null' retirar ela (caso não, vai deixar todos os valores como '-1')
    """
    # NORMAL FLOW
    result_df = database

    # Tratar valores infinitos
    result_df.loc[result_df[column] == np.inf, column] = result_df.loc[result_df[column] == np.inf, column].replace([np.inf], np.nan) # substituir valors infinitos por nan
    
    media_values = result_df.loc[result_df[column].notnull(),column].mean() # resgatar a média entre os valores dos dados normais
    
    if(analysis_method=="auto"):
        x = (len(result_df[result_df[column].isna()]))
        total = len(result_df[column])
        
        if(x/total <= (5/100) and important_column):
            result_df = result_df.drop(index=result_df[result_df[column].isna()].index) # retirar linhas com valores Nan  
        elif(x/total <= (40/100)):
            result_df.loc[result_df[column].isna(), column] = media_values # substituir valores encontrados pela média
        else:
            if(drop_column):
                result_df = result_df.drop(column, axis=1) # retirar coluna
            else:
                result_df[column] = -1 # substituir toda coluna por -1
    elif(analysis_method=="drop"):
        result_df = result_df.drop(index=result_df[result_df[column].isna()].index) # retirar valores Nan  
    elif(analysis_method=="avg"):
        result_df.loc[result_df[column].isna(), column] = media_values # substituir pela média
    elif(analysis_method=="zero"):
        result_df.loc[result_df[column].isna(), column] = 0 # substituir por zero
        
    
    return result_df

# Método para tratar os valores zerados e negativos das colunas passadas do dataframe
def remove_zero_and_negative(database, column, type=1, analysis_method="auto", important_column=False, drop_column=True):
    """ _summary_
    
    Args: 
        database (table): base de dados que se deseja tratar
        column (str): nome da coluna a ser tratada
        dropValues (bool): remover valores 
        typeRemove (int): 1 - both, 2 - negative, 3 - zero
        analysis_method (str): avg - média, drop - retirar linha, auto - automático
        important_column (bool): - irá retirar toda a linha se mais de 5% dos dados da coluna for '0' ou 'negativo' e se o valor for maior que 40% então substituir pela média
        drop_column (bool): caso a coluna seja maioria '0' ou 'negativo' retirar ela (caso não, vai deixar todos os valores como '-1')
    """
    # NORMAL FLOW
    
    result_df = database

    # Tratar valores zerados ou negativos
    if(type==1):
        result_df.loc[result_df[column] <= 0, column] = result_df.loc[result_df[column] <= 0, column] = np.nan # substituir valores negativos e zerados
    elif(type==2):
        result_df.loc[result_df[column] < 0, column] = result_df.loc[result_df[column] < 0, column] = np.nan # substituir valores negativos
    elif(type==3):
        result_df.loc[result_df[column] == 0, column] = result_df.loc[result_df[column] == 0, column] = np.nan # substituir valores zerados
    
    media_values = result_df.loc[result_df[column].notnull(),column].mean() # resgatar a média entre os valores dos dados normais
    
    if(analysis_method=="auto"):
        x = (len(result_df[result_df[column].isna()]))
        total = len(result_df[column])
        
        if(x/total <= (5/100) and important_column):
            result_df = result_df.drop(index=result_df[result_df[column].isna()].index) # retirar linhas
        elif(x/total <= (40/100)):
            result_df.loc[result_df[column].isna(), column] = media_values # substituir pela média
        else:
            if(drop_column):
                result_df = result_df.drop(column, axis=1) # retirar coluna
            else:
                result_df[column] = -1 # substituir toda coluna por -1
    elif(analysis_method=="drop"):
        result_df = result_df.drop(index=result_df[result_df[column].isna()].index) # retirar valores Nan  
    elif(analysis_method=="avg"):
        result_df.loc[result_df[column].isna(), column] = media_values # substituir pela média
    elif(analysis_method=="zero"):
        result_df.loc[result_df[column].isna(), column] = 0 # substituir pela
        
    return result_df

# Remover colunas duplicadas do dataframe (colunas com sufixo '.1')
def remove_duplicate_columns(database):
    # Definir dados
    result_df = database
    
    for col in database.columns:
        if col.endswith('.1'):
            result_df = result_df.drop(col, axis=1)
    
    return database

# Limpar espaços em branco no nome das colunas do dataframe
def limpar_espacos_colunas(df):
    df.columns = df.columns.str.strip()
    return df

# regatar colunas totalmente zeradas do dataframe
def get_columns_100_perc_zero(path, dataBases):
    # definir dados
    lenChunk = 1691471
    result = ["tmp"]

    for nameDataBase in dataBases:
        # Inicializa um dicionário para contar zeros por coluna
        contador_zeros = None
        total_linhas = 0
        arq = path + "/" + nameDataBase + ".csv"
        
        for chunk in pd.read_csv(arq, chunksize=lenChunk):
            
            total_linhas += len(chunk)

            # Conta zeros em cada coluna (True para zero, False para outros)
            zeros_por_col = (chunk == 0).sum()

            # Acumula a contagem
            if contador_zeros is None:
                contador_zeros = zeros_por_col
            else:
                contador_zeros += zeros_por_col

        # Verifica colunas com 100% de zeros
        colunas_100_zero = contador_zeros[contador_zeros == total_linhas].index.tolist()
        remove_dpl = set(colunas_100_zero + result) # remover colunas já encontradas# remover colunas já encontradas
    
        result = list(remove_dpl)
            
    return result   

##### Funções para tratar a base de dados


In [ ]:
def create_final_database(path, dataBases, columnsForDrop, finalPath="./files/csv/intermediary", finalArq="database.csv", ):
    # definir dados
    chunk_size = 1691471
    result_arq = path + "/" + finalArq
     
    # remover arquivo caso exista
    if(os.path.isfile(result_arq)):
        os.remove(result_arq);
    
    for db in dataBases:
        arq = path + "/" + db + ".csv"; 
        for chunk in pd.read_csv(arq, chunksize=chunk_size):
            chunk = limpar_espacos_colunas(chunk)
            chunk = chunk.drop(columnsForDrop, axis=1)
            chunk = drop_database_null_infinity_values(chunk)
            
            # criar arquivo caso não exista ou adicionar ao arquivo caso já exista
            if(os.path.isfile(result_arq)):
                chunk.to_csv(result_arq, mode='w', index=False, header=False) 
            else:
                chunk.to_csv(result_arq, index=False) 
        

#### Executar Pré-processamento

In [ ]:
# regatar colunas a serem removidas do dataframe
columns_for_remove = list(set(get_columns_100_perc_zero(pathDataTest, first_part + second_part) + columns_for_remove))
columns_for_remove = [col.strip() for col in columns_for_remove]

C:\Users\mathe\AppData\Local\Temp\ipykernel_3880\628499244.py:133: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(arq, chunksize=lenChunk):
C:\Users\mathe\AppData\Local\Temp\ipykernel_3880\628499244.py:133: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(arq, chunksize=lenChunk):
C:\Users\mathe\AppData\Local\Temp\ipykernel_3880\628499244.py:133: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(arq, chunksize=lenChunk):
C:\Users\mathe\AppData\Local\Temp\ipykernel_3880\628499244.py:133: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(arq, chunksize=lenChunk):
C:\Users\mathe\AppData\Local\Temp\ipykernel_3880\628499244.py:133: DtypeWarning: Columns (85) have mixed types. Spec

In [28]:
create_final_database(pathDataTest, first_part, columns_for_remove, finalPath=pathIntermediary,finalArq="preProcess_database_first_part.csv")

./files/csv/03-11/MSSQL.csv
./files/csv/03-11/Portmap.csv
./files/csv/03-11/Syn.csv
./files/csv/03-11/UDP.csv
./files/csv/03-11/UDPLag.csv
